In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()

PROJECT_ROOT = next(
    (
        path for path in [cwd, *cwd.parents]
        if (path / "data" / "raw" / "ai4i2020.csv").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/ai4i2020.csv."
    )

METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

threshold_path = METRICS_DIR / "threshold_analysis.csv"
oof_path = METRICS_DIR / "oof_model_comparison.csv"

threshold_df = pd.read_csv(threshold_path)
oof_comparison = pd.read_csv(oof_path)

print("Threshold results:", threshold_df.shape)
print("OOF comparison:", oof_comparison.shape)

Threshold results: (38, 11)
OOF comparison: (2, 3)


In [2]:
MIN_RECALL = 0.85
SELECTED_MODEL = "Hist_Gradient_Boosting"
SELECTED_THRESHOLD = 0.25

eligible = threshold_df[
    (threshold_df["model"] == SELECTED_MODEL)
    & (threshold_df["recall"] >= MIN_RECALL)
].copy()

best_candidate = eligible.sort_values(
    ["precision", "threshold"],
    ascending=[False, False]
).iloc[0]

print("Best tested eligible threshold:")
print(best_candidate[
    [
        "model",
        "threshold",
        "precision",
        "recall",
        "f1",
        "TP",
        "FP",
        "FN"
    ]
])

assert best_candidate["threshold"] == SELECTED_THRESHOLD
print("\nSelected threshold verified.")

Best tested eligible threshold:
model        Hist_Gradient_Boosting
threshold                      0.25
precision                  0.445507
recall                     0.859779
f1                         0.586902
TP                              233
FP                              290
FN                               38
Name: 4, dtype: object

Selected threshold verified.


In [3]:
threshold_decision = {
    "selected_model": SELECTED_MODEL,
    "selected_threshold": SELECTED_THRESHOLD,
    "selection_data": "Training out-of-fold predictions",
    "minimum_recall_constraint": MIN_RECALL,
    "selection_rule": (
        "Maximize precision among tested thresholds "
        "with recall >= 0.85"
    ),
    "imbalance_strategy": "Class weighting",
    "smote_used": False,
    "random_oversampling_used": False,
    "random_undersampling_used": False,
    "test_set_used_for_threshold_selection": False,
    "oof_precision": float(best_candidate["precision"]),
    "oof_recall": float(best_candidate["recall"]),
    "oof_f1": float(best_candidate["f1"]),
    "oof_true_positives": int(best_candidate["TP"]),
    "oof_false_positives": int(best_candidate["FP"]),
    "oof_false_negatives": int(best_candidate["FN"])
}

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

decision_path = REPORTS_DIR / "threshold_decision.json"

with open(decision_path, "w") as f:
    json.dump(threshold_decision, f, indent=4)

print("Threshold decision saved to:", decision_path)

Threshold decision saved to: d:\robotic-predictive-maintenance\artifacts\reports\threshold_decision.json
